# Notebook 01 — Modelos LLM e NLP com Hugging Face

**Objetivo:** Demonstrar dominio do ecossistema Hugging Face com tarefas NLP aplicadas ao dominio de bulas medicas, seguindo o estilo do professor (`pipeline`, `AutoTokenizer`, `AutoModel`).

**Rubrica 1:** Construir aplicacoes NLP com LLMs e ecossistema Hugging Face (5 itens).

## 2.1 Setup e Imports

In [1]:
import torch
from transformers import pipeline, AutoModel, AutoTokenizer
from scripts.config import DEVICE, NER_MODEL, EMBEDDING_MODEL

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponivel: {torch.cuda.is_available()}")
print(f"Device configurado: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.6.0+cu124
CUDA disponivel: True
Device configurado: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.4 GB


## 2.2 Carregando Modelo com AutoModel + AutoTokenizer

Demonstracao no estilo do professor: carregar um modelo pre-treinado, tokenizar entrada, e inspecionar as dimensoes dos hidden states.

**Modelo:** `pucpr/clinicalnerpt-chemical` — BERT treinado para NER em textos clinicos em portugues.

### Por que comecar com AutoModel?

`AutoModel.from_pretrained()` carrega o corpo do modelo (encoder) **sem cabecalho de tarefa** — util para entender a arquitetura antes de adicionar classificadores. As dimensoes `[Batch, Tokens, Hidden_Dim]` revelam:
- **Batch:** quantas frases processadas de uma vez
- **Tokens:** quantos tokens a tokenizacao gerou (incluindo `[CLS]` e `[SEP]`)
- **Hidden_Dim:** tamanho do embedding interno (768 para BERT base)

In [2]:
model_id = NER_MODEL  # "pucpr/clinicalnerpt-chemical"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).to(DEVICE)

# Processando a entrada (estilo do professor)
inputs = tokenizer("O mecanismo de atencao e poderoso", return_tensors="pt")
# Move para GPU
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
outputs = model(**inputs)

print(f"Dimensoes do output: {outputs.last_hidden_state.shape}")
print(f"Interpretacao: [Batch={outputs.last_hidden_state.shape[0]}, Tokens={outputs.last_hidden_state.shape[1]}, Hidden_Dim={outputs.last_hidden_state.shape[2]}]")

# Mostrar os tokens gerados
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(f"\nTokens: {tokens}")
print(f"Total de tokens: {len(tokens)}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

[transformers] BertModel LOAD REPORT from: pucpr/clinicalnerpt-chemical
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Dimensoes do output: torch.Size([1, 10, 768])
Interpretacao: [Batch=1, Tokens=10, Hidden_Dim=768]

Tokens: ['[CLS]', 'o', 'mecanismo', 'de', 'at', '##en', '##cao', 'e', 'poderoso', '[SEP]']
Total de tokens: 10


**Observacoes:**
- O tokenizador BERT usa **WordPiece**: palavras frequentes viram tokens unicos, palavras raras sao quebradas em sub-tokens
- O limite padrao e **512 tokens** — textos mais longos precisam de estrategias de truncamento ou chunking
- `last_hidden_state` contem o embedding contextualizado de cada token — diferente de embeddings estaticos (Word2Vec), estes variam conforme o contexto da frase